In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

In [ ]:
PATH = '../P3_Result/01.CSV/'

### 결과데이터 전처리

In [ ]:
#행정동별 최종 결과 데이터 프레임(지표 + 서비스권역내 결과)
RESULT_hjd = pd.read_csv(PATH + 'A22_읍면동별_지표산출결과.csv',encoding='cp949')
RESULT_hjd_service = pd.read_excel(PATH +'서비스권역내인구비율/서비스권역내_인구비율결과(행정동별).xlsx')
RESULT_hjd_service = RESULT_hjd_service.drop(['hjd_cd', 'geometry'], axis=1)
RESULT_hjd_service = RESULT_hjd_service.rename(columns={'hjd_nm': '행정기관'})
result_df_hjd = pd.merge(RESULT_hjd,RESULT_hjd_service,on='행정기관')

In [ ]:
#권역별 최종결과 데이터 프레임(지표 + 서비스권역내 결과)
RESULT_rei = pd.read_csv(PATH + 'A22_권역별_지표산출결과.csv',encoding='cp949')
RESULT_rei_service = pd.read_excel(PATH +'서비스권역내인구비율/서비스권역내_인구비율결과(권역별).xlsx')
RESULT_rei_service = RESULT_rei_service.drop(['geometry'], axis=1)
result_df_rei = pd.merge(RESULT_rei,RESULT_rei_service,on='권역명')

### 지니지수 산출
 - 행정동별만 진행

In [ ]:
#데이터 전처리
# 실제값이 양의값과 음의값을 보이므로 지니계수 도출에 적합하지 않다고 판단 -> 지니계수 산출대상에서 제외
hjd_filtered = result_df_hjd.drop(columns=['행정기관','인구증가율(%)', '사업체수 증감율(%)', '종사자수 증감율(%)'])
gini_hjd = hjd_filtered.copy()
gini_hjd

In [ ]:
#지니지수 산출 함수
def gini(df):
    gini_results = {}  # 컬럼 이름과 해당 컬럼의 지니 지수를 저장할 딕셔너리

    # 각 컬럼 반복
    for column in df.columns:
        # 숫자형변환
        values = pd.to_numeric(df[column], errors='coerce')

        # 숫자변환이 안되는 값은 제거
        #values = values.dropna()

        # 지니 지수 계산
        # 값들을 정렬하여 sorted_values에 저장
        sorted_values = sorted(values)

        # 높이(height)와 면적(area) 변수 초기화
        height, area = 0, 0

        # 정렬된 값들을 순회하면서 높이와 면적 계산
        for value in sorted_values:
            # 현재 값을 높이에 더함
            height += value
            # 면적에는 (높이 - 현재 값 / 2)를 더함
            area += height - value / 2.

        # 공정한 면적(fair_area) 계산
        fair_area = height * len(values) / 2.

        # 만약 공정한 면적(fair_area)이 0인 경우, 모든 값이 동일하여 완전한 평등을 나타냄
        # 이 경우, 지니 지수는 0으로 설정
        if fair_area == 0:
            gini_index = 0
        # 그렇지 않은 경우, (공정한 면적 - 면적) / 공정한 면적을 계산하여 지니 지수 구함
        else:
            gini_index = (fair_area - area) / fair_area

        # 소수점 2자리까지 반올림하여 저장
        gini_index = round(gini_index, 2)

        gini_results[column] = gini_index

    return gini_results


In [ ]:
re_gini_hjd = gini(gini_hjd)
re_gini_hjd = pd.DataFrame.from_dict(re_gini_hjd, orient='index', columns=['지니 지수'])
re_gini_hjd = re_gini_hjd.reset_index()
re_gini_hjd.columns  = ['항목명','지니지수']

In [ ]:
re_gini_hjd

In [ ]:
# 부문 값을 채우기 위한 함수 정의
def get_category(value):
    if value in ['총인구','세대수','인구증가율(%)', '출생률(%)', '노령화지수', '인구밀도(명/km²)' ,'기초생활수급자 비율(%)','65세이상 1인가구 비율(%)']:
        return '인구구조'
    elif value in ['사업체수' ,'종사자수' ,'사업체 1개소당 종사자수(명/개소)', '지방소득세 부과액(억원)','유동인구' ,'월평균_카드매출액(원)', '대지비율(%)', '평균공시지가(만원)' ,'사업체수 증감율(%)', '종사자수 증감율(%)','상용근로자 비율(%)', '상업지역비율(%)']:
        return '경제활력'
    elif value in ['유아(0~5세)_1,000명당_보육시설수','학령인구당_학교수(초중고)','어린이집 및 유치원 서비스권역 내  영유아인구 비율(750m)', '초등학교 서비스권역 내 학령인구 비율(750m)']:
        return '교육'
    elif value in ['인구1000명당_공공문화시설수', '인구1000명당_체육시설수', '인구1000명당_공원수', '녹지율(%)', '상수도보급률(%)','도시가스보급률(%)', '악취배출시설 수', '미세먼지_초미세먼지','문화시설 서비스권역 내 인구비율(10km)', '도서관 서비스권역 내 인구 비율(750m)','체육시설 서비스권역 내 인구 비율(750m)','공원 서비스권역 내 인구 비율(750m)']:
        return '생활환경'
    elif value in ['인구1000명당_의료시설수' ,'인구1000명당_병상수','인구1000명당_사회복지시설수','병원 서비스권역 내 인구비율(750m)', '노인복지시설 서비스권역 내 인구비율(750m)','사회복지시설 서비스권역 내 인구비율(750m)']:
        return '보건복지'
    elif value in ['노후주택비율','빈집비율','인구1000명당_교통사고건수','인구1000명당_자동차등록건수','소방서 서비스권역 내 인구 비율(7.5km)','경찰서 서비스권역 내 인구 비율(7.5km)']:
        return '안전'
    elif value in ['인구100명당_버스이용건수(건/100명)','도로집행율(%)','대중교통편의성','버스정류장 서비스권역 내 인구 비율(300m)']:
        return '교통'
    else:
        return None

In [ ]:
# '부문' 컬럼 추가 및 값 채우기
re_gini_hjd['부문'] = re_gini_hjd['항목명'].apply(get_category)
re_gini_hjd = re_gini_hjd[['부문','항목명','지니지수']]
re_gini_hjd

In [ ]:
re_gini_hjd.to_csv("C://j_code//2023//A22//P3_Result//01.CSV//A22_항목별_지니지수산출결과.csv",encoding='cp949',index = False)

### 읍면동별 /권역별 불균형 수준
 #### 1) 부문별 점수 및 종합점수 : 11개 읍면동, 52개 항목에 대하여 정규화한 측정치를 7개부문(인구구조,경제활력,교육,생활환경,보건복지,안전,교통)별로 종합하여 평균값을 구해 부문별 점수를 도출
   ###### ※ 서비스권역 4개 제외(국토부에서 다운받은 4개지표는 산출값이 없이 단순시각화만 진행하여 제외됨)

In [ ]:
#표준화 함수 생성

def apply_minmax_scaler(df):
    scaler = MinMaxScaler()
    columns_to_scale = df.columns[1:]  # 첫 번째 컬럼 제외

    for column in columns_to_scale:
        if df[column].nunique() == 1:
            # 특정 컬럼의 값이 모두 같은 경우 (문화시설,경찰서,소방서의 경우 실제값이 모두 100%로 동일함)
            df[column] = "{:.2f}".format(1)
        else:
            # MinMaxScaler 적용
            df[column] = scaler.fit_transform(df[[column]])
            df = df.round(2)
    return df

In [ ]:
#행렬변환 함수

def transpose_dataframe(df):
    df_trans = df.transpose()
    df_trans.columns = df_trans.iloc[0]
    df_trans = df_trans.iloc[1:].reset_index()
    df_trans = df_trans.rename(columns={'index': '항목명'})
    return df_trans

### 실제값 결과 저장

In [ ]:
# 실제값-> 행렬변환
result_hjd_trans = transpose_dataframe(result_df_hjd)
result_rei_trans = transpose_dataframe(result_df_rei)

In [ ]:
# '부문' 컬럼 추가 및 값 채우기
result_hjd_trans['부문'] = result_hjd_trans['항목명'].apply(get_category)
# '부문' 컬럼 추가 및 값 채우기
result_rei_trans['부문'] = result_rei_trans['항목명'].apply(get_category)

In [ ]:
# 부문 컬럼을 가장 앞으로 이동
cols = list(result_hjd_trans.columns)
cols.insert(0, cols.pop(cols.index('부문')))
result_hjd_trans = result_hjd_trans[cols]

# 부문 컬럼을 가장 앞으로 이동
cols2 = list(result_rei_trans.columns)
cols2.insert(0, cols2.pop(cols2.index('부문')))
result_rei_trans = result_rei_trans[cols2]

In [ ]:
# '부문' 컬럼을 기준으로 데이터프레임 정렬
result_hjd_trans = result_hjd_trans.sort_values(by='부문')
result_rei_trans = result_rei_trans.sort_values(by='부문')

In [ ]:
#A22 읍면동/권역별 지표산출결과
#result_hjd_trans.to_csv("C://j_code//2023//A22//P3_Result//01.CSV//A22_읍면동별_지표산출결과(최종).csv",encoding='cp949',index = False)
#result_rei_trans.to_csv("C://j_code//2023//A22//P3_Result//01.CSV//A22_권역별_지표산출결과(최종).csv",encoding='cp949',index = False)

In [ ]:
#result_df_hjd.to_csv("C://j_code//2023//A22//P3_Result//01.CSV//A22_읍면동별_지표산출결과.csv",encoding='cp949',index = False)
#result_df_rei.to_csv("C://j_code//2023//A22//P3_Result//01.CSV//A22_권역별_지표산출결과.csv",encoding='cp949',index = False)

### 표준화값 저장

In [ ]:
#실제갑 -> 표준화
df_hjd_scaled = apply_minmax_scaler(result_df_hjd)
df_rei_scaled = apply_minmax_scaler(result_df_rei)
# 표준화-> 행렬변환
result_hjd_scaled_trans = transpose_dataframe(df_hjd_scaled)
result_rei_scaled_trans = transpose_dataframe(df_rei_scaled)

In [ ]:
#df_hjd_scaled.to_csv("C://j_code//2023//A22//P3_Result//01.CSV//A22_읍면동별_표준화.csv",encoding='cp949',index = False)
#df_rei_scaled.to_csv("C://j_code//2023//A22//P3_Result//01.CSV//A22_권역별_표준화.csv",encoding='cp949',index = False)

In [ ]:
# '부문' 컬럼 추가 및 값 채우기
result_hjd_scaled_trans['부문'] = result_hjd_scaled_trans['항목명'].apply(get_category)
# '부문' 컬럼 추가 및 값 채우기
result_rei_scaled_trans['부문'] = result_rei_scaled_trans['항목명'].apply(get_category)

In [ ]:
# 부문 컬럼을 가장 앞으로 이동
cols = list(result_hjd_scaled_trans.columns)
cols.insert(0, cols.pop(cols.index('부문')))
result_hjd_scaled_trans = result_hjd_scaled_trans[cols]

In [ ]:
# 부문 컬럼을 가장 앞으로 이동
cols2 = list(result_rei_scaled_trans.columns)
cols2.insert(0, cols2.pop(cols2.index('부문')))
result_rei_scaled_trans = result_rei_scaled_trans[cols2]

In [ ]:
# '부문' 컬럼을 기준으로 데이터프레임 정렬
result_hjd_scaled_trans = result_hjd_scaled_trans.sort_values(by='부문')
result_rei_scaled_trans = result_rei_scaled_trans.sort_values(by='부문')

In [ ]:
#A22 읍면동/권역별 지표산출 표준화결과
result_hjd_scaled_trans.to_csv("C://j_code//2023//A22//P3_Result//01.CSV//A22_읍면동별_지표산출 표준화결과.csv",encoding='cp949',index = False)
result_rei_scaled_trans.to_csv("C://j_code//2023//A22//P3_Result//01.CSV//A22_권역별_지표산출 표준화결과.csv",encoding='cp949',index = False)

In [ ]:
pre_hjd_scaled_trans = result_hjd_scaled_trans.copy()
pre_rei_scaled_trans = result_rei_scaled_trans.copy()

In [ ]:
specific_values = ['악취배출시설 수', '미세먼지_초미세먼지', '인구1000명당_교통사고건수','빈집비율','노후주택비율','65세이상 1인가구 비율(%)','기초생활수급자 비율(%)','노령화지수']
numeric_columns = pre_hjd_scaled_trans.select_dtypes(include='number').columns #읍면동
numeric_columns2 = pre_rei_scaled_trans.select_dtypes(include='number').columns #권역별
pre_hjd_scaled_trans.loc[pre_hjd_scaled_trans['항목명'].isin(specific_values), numeric_columns] *= -1
pre_rei_scaled_trans.loc[pre_rei_scaled_trans['항목명'].isin(specific_values), numeric_columns2] *= -1
pre_rei_scaled_trans

### 읍면동별 불균형 수준

In [ ]:
# 특정 컬럼을 float로 변환
columns_to_convert = ['백석읍', '은현면', '남면', '광적면', '장흥면', '양주1동', '양주2동', '회천1동','회천2동', '회천3동', '회천4동']
pre_hjd_scaled_trans[columns_to_convert] = pre_hjd_scaled_trans[columns_to_convert].astype(float)

# 특정 컬럼을 float로 변환
columns_to_convert = ['1번권역', '2번권역', '3번권역', '4번권역']
pre_rei_scaled_trans[columns_to_convert] = pre_rei_scaled_trans[columns_to_convert].astype(float)

In [ ]:
result_hjd_rank = pre_hjd_scaled_trans[['부문','백석읍', '은현면', '남면', '광적면', '장흥면', '양주1동', '양주2동', '회천1동','회천2동', '회천3동', '회천4동']]
# 부문 컬럼을 기준으로 그룹화하여 각 그룹의 평균 값 계산
hjd_group = result_hjd_rank.groupby(['부문']).mean().round(2)
hjd_group

In [ ]:
result_rei_rank = pre_rei_scaled_trans[['부문','1번권역', '2번권역', '3번권역', '4번권역']]
# 부문 컬럼을 기준으로 그룹화하여 각 그룹의 평균 값 계산
rei_group = result_rei_rank.groupby(['부문']).mean().round(2)
rei_group

### 종합평균 산출

In [ ]:
hjd_all_mean = hjd_group.mean().round(2).reset_index()
hjd_all_mean.rename(columns={0: '종합평균'}, inplace=True)
hjd_all_mean = hjd_all_mean.transpose()
# 첫 번째 행을 컬럼으로 옮기기
hjd_all_mean.columns = hjd_all_mean.iloc[0]
hjd_all_mean = hjd_all_mean[1:]
hjd_all_mean

In [ ]:
rei_all_mean = rei_group.mean().round(2).reset_index()
rei_all_mean.rename(columns={0: '종합평균'}, inplace=True)
rei_all_mean = rei_all_mean.transpose()
# 첫 번째 행을 컬럼으로 옮기기
rei_all_mean.columns = rei_all_mean.iloc[0]
rei_all_mean = rei_all_mean[1:]
rei_all_mean

### 최종 불균형 지표 생성

In [ ]:
# 데이터프레임을 위아래로 합치기
result_hjd_unbal = pd.concat([hjd_all_mean, hjd_group], axis=0)

# 첫 번째 데이터프레임의 컬럼만 남기기
selected_columns = ['백석읍', '은현면', '남면', '광적면', '장흥면', '양주1동', '양주2동', '회천1동', '회천2동','회천3동', '회천4동']  # 남길 컬럼명을 리스트로 지정
final_df = result_hjd_unbal[selected_columns]
final_df_hjd = final_df.reset_index()
final_df_hjd.rename(columns={'index': '부문'}, inplace=True)
final_df_hjd

In [ ]:
# 데이터프레임을 위아래로 합치기
result_rei_unbal = pd.concat([rei_all_mean, rei_group], axis=0)

# 첫 번째 데이터프레임의 컬럼만 남기기
selected_columns = ['1번권역', '2번권역', '3번권역', '4번권역']  # 남길 컬럼명을 리스트로 지정
final_df = result_rei_unbal[selected_columns]
final_df_rei = final_df.reset_index()
final_df_rei.rename(columns={'index': '부문'}, inplace=True)
final_df_rei

In [ ]:
#A22 읍면동/권역별 불균형점수
final_df_hjd.to_csv("C://j_code//2023//A22//P3_Result//01.CSV//A22_읍면동별_불균형점수2.csv",encoding='cp949',index = False)
final_df_rei.to_csv("C://j_code//2023//A22//P3_Result//01.CSV//A22_권역별_불균형점수2.csv",encoding='cp949',index = False)